<a href="https://colab.research.google.com/github/stephanie465337/Data-Science-Portfolio-C21/blob/main/NotebookLM/Module%201/8d_Pandas_Wide_to_Long.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Going from wide to long


In data analysis, unpivoting (often referred to as "melting") is the process of transforming a dataset from a wide format—where data is spread across multiple columns—into a long (or narrow) format, where those column headers become values in a single row. This is a crucial step for "tidying" data, as most modern visualization tools and statistical models prefer one observation per row and one variable per column.

In Pandas, this is primarily handled by the `melt()` method. When you unpivot, you distinguish between identifier variables (columns you want to keep as-is, like 'Name' or 'Date') and measured variables (the columns you want to collapse). The function then stacks the measured columns into two new columns: one containing the original column names (the "variable") and another containing the values themselves.

Another powerful option is to use the `.stack()` method, which approaches the problem from a structural perspective rather than a column-naming one. While `melt()` "liquefies" your data by targeting specific columns, `stack()` works by "compressing" the existing column headers down into the row index. It effectively pivots the innermost level of the column labels to become the innermost level of the row index, resulting in a MultiIndex Series, which you can then reset into a data frame.



From https://stackoverflow.com/a/48136002


## Setup



In [ ]:
from io import StringIO
import pandas as pd
import markdown
import duckdb

### Convert between data formats

#### Markdown/Text


In [ ]:
#Assign Markdown string to a variable
table_md = '''
|Date|A|B|C|D|E |
|----|-|-|-|-|--|
|2005|1|2|3|4|50|
|2006|6|7|8|9|10|
'''
table_md

'\n|Date|A|B|C|D|E |\n|----|-|-|-|-|--|\n|2005|1|2|3|4|50|\n|2006|6|7|8|9|10|\n'

In [ ]:
print(table_md)


|Date|A|B|C|D|E |
|----|-|-|-|-|--|
|2005|1|2|3|4|50|
|2006|6|7|8|9|10|



#### HTML


In [ ]:
#Convert Markdown to HTML
table_html=markdown.markdown(table_md, extensions=['markdown.extensions.tables'])
table_html

'<table>\n<thead>\n<tr>\n<th>Date</th>\n<th>A</th>\n<th>B</th>\n<th>C</th>\n<th>D</th>\n<th>E</th>\n</tr>\n</thead>\n<tbody>\n<tr>\n<td>2005</td>\n<td>1</td>\n<td>2</td>\n<td>3</td>\n<td>4</td>\n<td>50</td>\n</tr>\n<tr>\n<td>2006</td>\n<td>6</td>\n<td>7</td>\n<td>8</td>\n<td>9</td>\n<td>10</td>\n</tr>\n</tbody>\n</table>'

In [ ]:
%%html
<table>
<thead>
<tr>
<th>Date</th>
<th>A</th>
<th>B</th>
<th>C</th>
<th>D</th>
<th>E</th>
</tr>
</thead>
<tbody>
<tr>
<td>2005</td>
<td>1</td>
<td>2</td>
<td>3</td>
<td>4</td>
<td>50</td>
</tr>
<tr>
<td>2006</td>
<td>6</td>
<td>7</td>
<td>8</td>
<td>9</td>
<td>10</td>
</tr>
</tbody>
</table>

#### Data Frame

In [ ]:
#Convert HTML to a DataFrame
df = pd.read_html(StringIO(table_html))[0]
df

,Date,A,B,C,D,E
0,2005,1,2,3,4,50
1,2006,6,7,8,9,10


## Using .stack()

#### "Protect" columns by converting to index

In [ ]:
(
  df
    .set_index('Date')
)

,A,B,C,D,E
Date,,,,,
2005,1,2,3,4,50
2006,6,7,8,9,10


#### Stack - converting columns to key-value pairs

In [ ]:
(
  df
    .set_index('Date')
    .stack()
)

Date   
2005  A     1
      B     2
      C     3
      D     4
      E    50
2006  A     6
      B     7
      C     8
      D     9
      E    10
dtype: int64

In [ ]:
(
  df
    .set_index('Date')
    .stack()
    .rename('Value')
)

Date   
2005  A     1
      B     2
      C     3
      D     4
      E    50
2006  A     6
      B     7
      C     8
      D     9
      E    10
Name: Value, dtype: int64

In [ ]:
(
  df
    .set_index('Date')
    .stack()
    .rename('Value')
    .rename_axis(['Date','Category'])
)

Date  Category
2005  A            1
      B            2
      C            3
      D            4
      E           50
2006  A            6
      B            7
      C            8
      D            9
      E           10
Name: Value, dtype: int64

In [ ]:
df_stack = (
  df
  .set_index('Date')
  .stack()
  .rename('Value')
  .rename_axis(['Date','Category'])
  .reset_index()
)
df_stack

,Date,Category,Value
0,2005,A,1
1,2005,B,2
2,2005,C,3
3,2005,D,4
4,2005,E,50
5,2006,A,6
6,2006,B,7
7,2006,C,8
8,2006,D,9
9,2006,E,10


## Using .melt()

### List of columns to "keep"

In [ ]:
keeps = df.columns[:1]
keeps

Index(['Date'], dtype='object')

### Melt


In [ ]:
df_melt = df.melt(
  keeps,
  var_name='Category',
  value_name='Val'
)
df_melt

,Date,Category,Val
0,2005,A,1
1,2006,A,6
2,2005,B,2
3,2006,B,7
4,2005,C,3
5,2006,C,8
6,2005,D,4
7,2006,D,9
8,2005,E,50
9,2006,E,10


### Sorted by "Date", then "Category"

In [ ]:
df_melt2 = (
  df
    .melt(
      keeps,
      var_name='Category',
      value_name='Val'
    )
    .sort_values(['Date','Category'])
    .reset_index(drop=True)
)
df_melt2

,Date,Category,Val
0,2005,A,1
1,2005,B,2
2,2005,C,3
3,2005,D,4
4,2005,E,50
5,2006,A,6
6,2006,B,7
7,2006,C,8
8,2006,D,9
9,2006,E,10


# Going from long to wide


Which process you use depends on the initial format and the desired final format.


## Using `.pivot()`

Pivoting takes at least three columns, converting the first into an index, the categories into the second as the columns, and the values in the third as the corresponding values for the columns.

We can use pivot to reverse the process.  Notice that we have to turn the "Date" column into an index.  

In [ ]:
(
df_melt2
  .pivot(
    index=['Date'],
    columns=['Category'],
    values='Val',
  )
)

Category,A,B,C,D,E
Date,,,,,
2005,1,2,3,4,50
2006,6,7,8,9,10


We can fix that by reseting and renaming the indices.

In [ ]:
(
df_melt2
  .pivot(
    index=['Date'],
    columns=['Category'],
    values='Val')
  .reset_index()
  .rename_axis(None, axis=1)
)

,Date,A,B,C,D,E
0,2005,1,2,3,4,50
1,2006,6,7,8,9,10


In [ ]:
ct = df.assign(fruit=df.fruit.str.split(", ")).explode("fruit").drop(columns=["id"])
ct

In [ ]:
ct["fruit"].unique()

In [ ]:
ct.value_counts()

In [ ]:
df.assign(fruit=df.fruit.str.split(", ")).explode("fruit").value_counts()

In [ ]:
df.assign(fruit=df.fruit.str.split(", ")).explode("fruit").value_counts().reset_index()

# Census


In [ ]:
url = "https://www2.census.gov/programs-surveys/acs/summary_file/2024/table-based-SF/data/5YRData/acsdt5y2024-b01001.dat"
url

'https://www2.census.gov/programs-surveys/acs/summary_file/2024/table-based-SF/data/5YRData/acsdt5y2024-b01001.dat'

In [ ]:
df = pd.read_csv(url, sep='|', nrows=None)
df.shape

(616690, 99)

In [ ]:
df.head(10)

,GEO_ID,B01001_E001,B01001_M001,B01001_E002,B01001_M002,B01001_E003,B01001_M003,B01001_E004,B01001_M004,B01001_E005,...,B01001_E045,B01001_M045,B01001_E046,B01001_M046,B01001_E047,B01001_M047,B01001_E048,B01001_M048,B01001_E049,B01001_M049
0,0100000US,334922499,-555555555,165808018,7054,9601376,4166,10286448,19025,10987385,...,5713148,16975,8208903,18138,5750350,13939,3775132,12194,4067600,13037
1,0100089US,1053712,13108,525720,7100,32581,1005,38201,1158,40854,...,19152,535,26710,791,18277,645,10782,482,11107,719
2,0100090US,1587,466,723,227,48,41,125,93,30,...,17,11,58,27,38,34,23,16,16,16
3,0100091US,2631117,2727,1312963,1965,81717,607,87221,1361,98137,...,44298,1270,65219,1266,45400,1006,30651,905,29307,1085
4,0100092US,55684,1155,27644,634,1029,116,1424,153,1508,...,1696,140,2525,170,1730,138,938,86,841,91
5,0100093US,276152,1233,145664,983,9151,172,9943,387,11398,...,4451,314,5618,336,3612,264,1751,174,1426,182
6,0100094US,1125534,6846,547703,4003,37822,891,39194,1313,39280,...,18750,650,26329,796,18576,676,12509,709,11981,645
7,0100095US,42212,2728,20899,1490,1479,258,1492,197,1581,...,838,128,938,120,689,85,542,101,478,119
8,01000A0US,317015024,615,156723909,6091,9113032,3752,9751672,18719,10408864,...,5338149,16874,7665307,17547,5366730,13739,3520413,11777,3805190,12781
9,01000C0US,288944434,377,142664231,4532,8329375,2896,8899472,17951,9492823,...,4791369,16298,6887190,16312,4821500,13415,3156435,11488,3432047,12626


In [ ]:
df_long = df.melt(
    id_vars = df.columns[:1],
    value_vars= df.columns[1:],
)
df_long.shape

In [ ]:
#For next time:
#^([^_]+)_([A-Z])(\d+)$

#DuckDB can see the 'df_long' variable in your local scope automatically
sql = """
SELECT
    *,
    regexp_extract(variable, '^(.+)_(.)(.+)$', 1) AS id_part,
    regexp_extract(variable, '^(.+)_(.)(.+)$', 2) AS type_letter,
    regexp_extract(variable, '^(.+)_(.)(.+)$', 3) AS number_part
FROM df_long
"""

#Execute and turn back into a parquet file
duckdb.query(sql).to_parquet("parsed_data.parquet")

In [ ]:
!ls -la --si

In [ ]:
duckdb.query("select * from parsed_data.parquet limit 10").df()

In [ ]:
sql = """
with sample as (
  select *
  FROM parsed_data.parquet
)

PIVOT (
    SELECT
        GEO_ID,
        id_part,
        number_part,
        type_letter,
        value
    FROM sample
)
ON type_letter
USING CAST(SUM(value) AS INTEGER)
GROUP BY GEO_ID, id_part, number_part
ORDER BY GEO_ID, id_part, number_part
"""

df_wide = duckdb.query(sql).df()
df_wide

In [ ]:
df_wide.info(memory_usage='deep')

In [ ]:
duckdb.query("""
    COPY (
        SELECT *,
               LEFT(split_part(GEO_ID, 'US', 2), 2) as state_fips
        FROM df_wide
    )
    TO 'census_data_partitioned'
    (FORMAT PARQUET, PARTITION_BY (state_fips, id_part), OVERWRITE_OR_IGNORE 1)
""")

In [ ]:
!ls -la

In [ ]:
!du -ms census_data_partitioned/

In [ ]:
duckdb.query('''
select
  GEO_ID,
  state_fips,
  id_part,
  number_part,
  E as estimate,
  M as margin_of_error
from read_parquet('census_data_partitioned/**/*.parquet', hive_partitioning=True)
limit 10
''')